[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-13-privacy-attacks-and-dp-sgd.ipynb)

# Advanced Discussion 13 — membership inference, canaries and clip-and-noise training

Loss-threshold membership inference, a planted canary, and a clip-and-noise (DP-SGD-style) trainer written with torch.func. No privacy accountant is included.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Uses scikit-learn's bundled digits; takes a few minutes on CPU.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.func import functional_call, vmap, grad
torch.manual_seed(0); np.random.seed(0)

# members = the small training set; non-members = data the model never saw
X, y = load_digits(return_X_y=True); X = X / 16.0
Xm, Xn, ym, yn = train_test_split(X, y, train_size=300, test_size=300, random_state=0, stratify=y)     # 300 members, 300 non-members
T = lambda a: torch.tensor(a, dtype=torch.float32)
Xm, Xn, ym, yn = T(Xm), T(Xn), torch.tensor(ym), torch.tensor(yn)
Xtest = T(X[~np.isin(np.arange(len(X)), [])][:0])                                                        # placeholder to keep names tidy
def make(): return nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 10))
def per_example_loss(m, a, b):
    with torch.no_grad(): return F.cross_entropy(m(a), b, reduction="none")
def mia_auc(m):                                          # loss-threshold attack: lower loss => more likely a member
    lm, ln = per_example_loss(m, Xm, ym), per_example_loss(m, Xn, yn)
    return roc_auc_score(np.r_[np.ones(len(lm)), np.zeros(len(ln))], -np.r_[lm.numpy(), ln.numpy()])
acc = lambda m, a, b: (m(a).argmax(1) == b).float().mean().item()

## 1. overfitting leaks membership

In [ ]:
def train(epochs, wd=0.0, seed=0):
    torch.manual_seed(seed); m = make(); o = torch.optim.Adam(m.parameters(), lr=2e-3, weight_decay=wd)
    for _ in range(epochs):
        perm = torch.randperm(len(Xm))
        for i in range(0, len(perm), 32):
            b = perm[i:i + 32]; o.zero_grad(); F.cross_entropy(m(Xm[b]), ym[b]).backward(); o.step()
    return m.eval()
print("training regime                 train acc  held-out acc  membership-inference AUC (0.5 = no leak)")
for name, ep, wd in (("early stop (5 epochs)", 5, 0.0), ("20 epochs", 20, 0.0), ("300 epochs (memorised)", 300, 0.0), ("300 epochs + weight decay 1e-2", 300, 1e-2)):
    m = train(ep, wd); print("%-31s %.3f      %.3f         %.3f" % (name, acc(m, Xm, ym), acc(m, Xn, yn), mia_auc(m)))

## 2. DP-SGD-style training: clip every example's gradient, add noise (no privacy accountant here)

In [ ]:
def dp_train(noise_mult, clip=1.0, epochs=60, seed=0, data=None):
    Xd, yd = data if data is not None else (Xm, ym)
    torch.manual_seed(seed); m = make(); params = {k: v.detach() for k, v in m.named_parameters()}
    def loss_fn(p, x, t): return F.cross_entropy(functional_call(m, p, (x.unsqueeze(0),)), t.unsqueeze(0))
    per_grad = vmap(grad(loss_fn), in_dims=(None, 0, 0))
    lr = 0.5
    for _ in range(epochs):
        perm = torch.randperm(len(Xd))
        for i in range(0, len(perm), 64):
            b = perm[i:i + 64]; g = per_grad(params, Xd[b], yd[b])
            norms = torch.sqrt(sum((v.flatten(1) ** 2).sum(1) for v in g.values()))
            scale = (clip / (norms + 1e-6)).clamp(max=1.0)                                  # per-example clipping
            for k in params:
                gk = (g[k] * scale.view(-1, *([1] * (g[k].dim() - 1)))).sum(0) + noise_mult * clip * torch.randn_like(params[k])
                params[k] = params[k] - lr * gk / len(b)
    with torch.no_grad():
        for k, v in m.named_parameters(): v.copy_(params[k])
    return m.eval()
print("\nDP-SGD-style (clip 1.0):  noise multiplier   held-out acc   MIA AUC")
for nm in (0.0, 0.5, 1.0, 2.0):
    m = dp_train(nm); print("                          %5.1f              %.3f         %.3f" % (nm, acc(m, Xn, yn), mia_auc(m)))

## 3. a risk register skeleton lives in the chapter text; here: extraction-style check for canaries

In [ ]:
# insert a rare 'canary' record into training and check whether the model is unusually sure about it
canary_x = T(np.random.RandomState(9).rand(1, 64)); canary_y = torch.tensor([3])
Xc, yc = torch.cat([Xm, canary_x]), torch.cat([ym, canary_y])
def train_c(epochs, seed=0):
    torch.manual_seed(seed); m = make(); o = torch.optim.Adam(m.parameters(), lr=2e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(Xc))
        for i in range(0, len(perm), 32):
            b = perm[i:i + 32]; o.zero_grad(); F.cross_entropy(m(Xc[b]), yc[b]).backward(); o.step()
    return m.eval()
print("\ncanary: a random-noise image labelled '3' added to the training set")
rand_imgs = T(np.random.RandomState(5).rand(200, 64))
for ep in (5, 300):
    m = train_c(ep)
    with torch.no_grad():
        p_canary = F.softmax(m(canary_x), 1)[0, 3].item(); p_random = F.softmax(m(rand_imgs), 1)[:, 3].mean().item()
    print("  %3d epochs: P('3' | canary) = %.3f vs mean P('3') on other random images = %.3f" % (ep, p_canary, p_random))

print("\nsame canary test under DP-SGD-style training (60 epochs, clip 1.0):")
for nm in (0.0, 1.0, 2.0):
    m = dp_train(nm, data=(Xc, yc))
    with torch.no_grad(): p_canary = F.softmax(m(canary_x), 1)[0, 3].item(); p_random = F.softmax(m(rand_imgs), 1)[:, 3].mean().item()
    print("  noise multiplier %.1f: P('3' | canary) = %.3f vs random images %.3f; held-out acc %.3f" % (nm, p_canary, p_random, acc(m, Xn, yn)))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.